- Parcial 2 : Análisis de datos del mercado laboral GEIH 2026 
- Fabian Andres Molina Arango

1. Para el total nacional calcule (a Junio de 2026): 
- -población en edad de trabajar  
- -población que no está en edad de trabajar 
- -población económicamente activa 
- -población económicamente inactiva 
- -tasa de ocupación 
- -tasa de desempleo  

In [0]:
#Leyendo la tabla de características generales del catálogo
cgse=spark.read.format("delta").table("workspace.electiva.cgse")
ft = spark.read.format("delta").table("workspace.electiva.ft")

-Población en edad de trabajar

In [0]:

PET = ft.filter(ft['PET'] == 1).agg({'FEX_C18': 'sum'}).collect()[0][0]
print(f'La población que está en edad de trabajar es de {PET:,.0f} personas')

-Población que no está en edad de trabajar

In [0]:
cgse = cgse.toPandas()
PT = cgse['FEX_C18'].sum()
PNET = PT - PET
print(f'La población que no está en edad de trabajar es de {PNET:,.0f} personas')

-Población económicamente activa

In [0]:

PEA = ft.filter(ft['FT'] == 1).agg({'FEX_C18': 'sum'}).collect()[0][0]
print(f'La población económicamente activa es de {PEA:,.0f} personas')

-Población económicamente inactiva

In [0]:

PEI = PET - PEA
print(f'La población económicamente inactiva es de {PEI:,.0f} personas')

-Tasa de ocupación

In [0]:

ocupados = spark.read.format("delta").table("workspace.electiva.ocupados").toPandas()
O = ocupados[ocupados['OCI']==1]['FEX_C18'].sum()
TO = O/PEA*100
TO = O/PET*100
print(f'El porcentaje de ocupados es de {TO:.2f}%')

-Tasa de desempleo

In [0]:

desempleados = spark.read.format("delta").table("workspace.electiva.desempleados").toPandas()
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()  # Cell 23
TD = D/PEA*100
print(f'El porcentaje de desempleados es de {TD:.2f}%')

Población en edad de trabajar (PET)
41.202.631 personas
La población en edad de trabajar corresponde a las personas que cumplen la edad establecida por la metodología estadística para formar parte de esta población. Este grupo constituye la base para analizar la participación en el mercado laboral. Sin embargo, estar en edad de trabajar no significa que una persona tenga empleo o esté buscando uno.

Población que no está en edad de trabajar
11.346.181 personas
Este resultado representa a las personas que todavía no cumplen la edad establecida para integrar la población en edad de trabajar. Se trata principalmente de niños y niñas. Esta población no se incluye en el cálculo de la PET ni en las tasas habituales de participación, ocupación y desempleo.

Población económicamente activa (PEA)
26.593.429 personas
La PEA reúne a las personas ocupadas y a las desocupadas que participan en el mercado laboral. Es decir, incluye tanto a quienes tienen trabajo como a quienes no tienen empleo, pero están disponibles y cumplen los criterios de búsqueda establecidos por la metodología. Este indicador permite conocer cuántas personas están vinculadas al mercado laboral mediante el empleo o la búsqueda activa de trabajo.

Población económicamente inactiva (PEI)
14.609.202 personas
La PEI está conformada por personas en edad de trabajar que no están ocupadas ni cumplen las condiciones para ser clasificadas como desocupadas. En este grupo pueden encontrarse estudiantes, personas dedicadas a labores del hogar, pensionados y otras personas que no participan activamente en el mercado laboral. Su condición no significa necesariamente que no realicen actividades útiles para la sociedad.

Tasa de ocupación (TO)
59,36%
La tasa de ocupación indica la proporción de personas ocupadas respecto a la población en edad de trabajar. Según el resultado obtenido, aproximadamente 59 de cada 100 personas que están en edad de trabajar se encuentran ocupadas. Este indicador permite evaluar la capacidad del mercado laboral para generar ocupación, aunque por sí solo no informa si los empleos son formales, estables o bien remunerados.

Tasa de desempleo (TD)
8,03%
La tasa de desempleo representa la proporción de personas desocupadas dentro de la población económicamente activa. El resultado indica que aproximadamente 8 de cada 100 personas que participan en el mercado laboral están desempleadas. Esto evidencia que, aunque una parte importante de la población participante tiene empleo, todavía hay personas que buscan trabajo y no lo consiguen.

2. Calcule las tasas de desempleo para hombres y mujeres en todos los departamentos de Colombia a Junio de 2026, ¿cuáles son los departamentos con mayores y menores tasas de desempleo?  

In [0]:
import pandas as pd

# Leyendo la tabla de características generales
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()

# Primero hay que anexar la P3271 a la tabla de desempleados
desempleados = pd.merge(
    desempleados,
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left'
)

ft = pd.merge(
    ft,
    cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271', 'P6040']],
    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'],
    how='left'
)

In [0]:
# desempleados_dpto -> cantidad de desempleados por departamento
desempleados_dpto = desempleados[desempleados['DSI'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

In [0]:

desempleados_dpto

In [0]:
# Cantidad de hombres y mujeres económicamente activos por departamento
pea_dpto_genero = ft[ft['FT'] == 1].groupby(['P3271', 'DPTO'])['FEX_C18'].sum()

In [0]:
tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero,'DESEMPLEADOS': desempleados_dpto})

In [0]:

tabla_desempleo

tabla_desempleo['TD'] = (
    tabla_desempleo['DESEMPLEADOS'] / tabla_desempleo['PEA']
) * 100
tabla_desempleo.head(100)

In [0]:
# 4. Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO',
    66: 'RISARALDA', 68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA',
    76: 'VALLE DEL CAUCA', 81: 'ARAUCA', 85: 'CASANARE', 86: 'PUTUMAYO',
    88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA', 95: 'GUAVIARE',
    97: 'VAUPES', 99: 'VICHADA'
}

sexo = {1: 'Hombre', 2: 'Mujer'}

In [0]:
tabla_desempleo = tabla_desempleo.reset_index()

tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo)

In [0]:
tabla_desempleo = tabla_desempleo[
    ['Departamento', 'Sexo', 'PEA', 'DESEMPLEADOS', 'TD']
]


In [0]:
# Ordenar por dos variables de mayor a menor (descendente)
tabla_desempleo = tabla_desempleo.sort_values(
    by=['Sexo', 'TD'],
    ascending=False
)

tabla_desempleo

In [0]:

tabla_desempleo[tabla_desempleo['Sexo'] == 'Mujer']
     

In [0]:

tabla_desempleo[tabla_desempleo['Sexo'] == 'Hombre']

El análisis de las tasas de desempleo por departamento permite observar diferencias en el comportamiento del mercado laboral colombiano según el territorio y el género. En varios departamentos, las mujeres presentan tasas de desempleo superiores a las de los hombres, lo que evidencia posibles desigualdades en las oportunidades de acceso al empleo. Sin embargo, este comportamiento no se presenta de la misma manera en todos los territorios.
Departamentos con mayores tasas de desempleo
Entre los resultados más destacados se encuentra Putumayo, donde la tasa de desempleo alcanza el 25,02 % para las mujeres y el 19,68 % para los hombres. También sobresale San Andrés, con una tasa del 18,16 % para las mujeres y del 20,98 % para los hombres, siendo este último porcentaje superior al registrado para las mujeres.
Por otra parte, Sucre y Bolívar presentan tasas elevadas de desempleo femenino, con un 20,49 % y un 17,40 %, respectivamente. En el caso de los hombres, también se destacan Arauca, con una tasa del 14,49 %, y Guaviare, con un 12,13 %.
Estos resultados muestran que las dificultades para conseguir empleo varían entre departamentos y que las diferencias entre hombres y mujeres no siguen un patrón uniforme. Por eso, es importante considerar las características económicas y productivas de cada territorio antes de establecer posibles explicaciones.
Departamentos con menores tasas de desempleo
En el extremo contrario, Vaupés registra las tasas de desempleo más bajas para ambos géneros, con un 1,78 % en las mujeres y un 2,99 % en los hombres. Amazonas también presenta porcentajes bajos, especialmente en el caso de las mujeres, cuya tasa es del 2,61 %.
Asimismo, Huila y Nariño registran tasas relativamente bajas para hombres y mujeres. En Risaralda, la tasa masculina es del 4,00 %, mientras que la femenina alcanza el 9,60 %, lo que refleja una diferencia importante entre ambos grupos.
No obstante, las tasas bajas deben interpretarse con precaución, especialmente en departamentos con poblaciones económicamente activas pequeñas, porque sus resultados pueden variar considerablemente y no necesariamente representan una mayor calidad del empleo.
Los resultados de junio de 2026 muestran que el desempleo presenta diferencias importantes entre departamentos y entre hombres y mujeres. Putumayo y San Andrés sobresalen por sus elevadas tasas en ambos grupos, mientras que Vaupés registra los porcentajes más bajos según los datos analizados. Además, en varios departamentos se observa una mayor tasa de desempleo femenino, aunque existen excepciones, como San Andrés, donde la tasa masculina es superior.
Estos resultados permiten reconocer que el mercado laboral colombiano tiene comportamientos distintos según las características de cada territorio. Sin embargo, para comprender las causas de estas diferencias, sería necesario complementar el análisis con información sobre informalidad, actividades económicas, ingresos y oportunidades laborales, teniendo en cuenta que una tasa de desempleo baja no garantiza, por sí sola, mejores condiciones de trabajo.

3. Tomando en consideración únicamente los siguientes grupos de edad:  
- 18 a 24
- 24 a 30
- 30 a 40
- 40 a 50
- 60 en adelante

Para cada uno de los departamentos, cual es el grupo de edad con mayor y menor tasa de desempleo ?  

In [0]:
# Función para mapear la edad (P6040) a grupos de edad
def mapear_grupo_edad(edad):
    if 18 <= edad < 24:
        return '18 a 24'
    elif 24 <= edad < 30:
        return '24 a 30'
    elif 30 <= edad < 40:
        return '30 a 40'
    elif 40 <= edad < 50:
        return '40 a 50'
    elif edad >= 60:
        return '60 en adelante'
    else:
        return None

# Aplicar el mapeo a ambos DataFrames
desempleados['GRUPO_EDAD'] = desempleados['P6040'].map(mapear_grupo_edad)
ft['GRUPO_EDAD'] = ft['P6040'].map(mapear_grupo_edad)
     

In [0]:
# Desempleados por grupo de edad y departamento (reutilizando código de pregunta 2)
desempleados_dpto_edad = desempleados[desempleados['DSI'] == 1].groupby(['GRUPO_EDAD', 'DPTO'])['FEX_C18'].sum()

# PEA por grupo de edad y departamento
pea_dpto_edad = ft[ft['FT'] == 1].groupby(['GRUPO_EDAD', 'DPTO'])['FEX_C18'].sum()

# Crear tabla de desempleo (reutilizando estructura de pregunta 2)
tabla_desempleo_edad = pd.DataFrame({'PEA': pea_dpto_edad, 'DESEMPLEADOS': desempleados_dpto_edad})

In [0]:
# Calcular tasa de desempleo (reutilizando código de pregunta 2)
tabla_desempleo_edad['TD'] = (tabla_desempleo_edad['DESEMPLEADOS'] / tabla_desempleo_edad['PEA']) * 100

# Mapear nombres de departamentos (reutilizando dpto_nombre de pregunta 2)
tabla_desempleo_edad = tabla_desempleo_edad.reset_index()
tabla_desempleo_edad['Departamento'] = tabla_desempleo_edad['DPTO'].map(dpto_nombre)

# Seleccionar y ordenar columnas
tabla_desempleo_edad = tabla_desempleo_edad[['Departamento', 'GRUPO_EDAD', 'PEA', 'DESEMPLEADOS', 'TD']]
tabla_desempleo_edad = tabla_desempleo_edad.sort_values(by=['Departamento', 'TD'], ascending=[True, False])

display(tabla_desempleo_edad)

In [0]:
# Grupo de edad con mayor tasa de desempleo por departamento
max_td = tabla_desempleo_edad.loc[tabla_desempleo_edad.groupby('Departamento')['TD'].idxmax()]
max_td = max_td[['Departamento', 'GRUPO_EDAD', 'TD']].reset_index(drop=True)
max_td.columns = ['Departamento', 'Mayor grupo de edad', 'Mayor TD']

# Grupo de edad con menor tasa de desempleo por departamento
min_td = tabla_desempleo_edad.loc[tabla_desempleo_edad.groupby('Departamento')['TD'].idxmin()]
min_td = min_td[['Departamento', 'GRUPO_EDAD', 'TD']].reset_index(drop=True)
min_td.columns = ['Departamento', 'Menor grupo de edad', 'Menor TD']

# Combinar resultados
resultado_edad = pd.merge(max_td, min_td, on='Departamento')
display(resultado_edad)

El análisis de las tasas de desempleo por edad y departamento permite identificar diferencias importantes en las oportunidades de acceso al mercado laboral. En términos generales, los jóvenes entre los 18 y 24 años presentan las mayores tasas de desempleo en la mayoría de los departamentos, mientras que las personas de 60 años en adelante registran las menores tasas en buena parte del territorio nacional.
Departamentos con mayores tasas de desempleo por edad
Entre los resultados más destacados se encuentran San Andrés y Sucre, donde la tasa de desempleo de los jóvenes entre 18 y 24 años alcanza el 33,66 % y el 33,14 %, respectivamente. Esto significa que aproximadamente uno de cada tres jóvenes económicamente activos de ese grupo de edad se encuentra desempleado en estos territorios.
También sobresalen Vichada, con una tasa del 31,60 % para los jóvenes de 18 a 24 años, y Bolívar, con un 30,86 % para ese mismo grupo. Por su parte, Putumayo presenta su mayor tasa en el grupo de 24 a 30 años, con un 30,58 %.
Estos resultados reflejan las dificultades que pueden enfrentar los jóvenes para ingresar al mercado laboral, especialmente durante la transición entre la educación y el empleo. Sin embargo, para establecer las causas concretas sería necesario analizar factores como la experiencia laboral exigida, la oferta de vacantes y las actividades económicas de cada departamento.
Departamentos con menores tasas de desempleo por edad
En contraste, las tasas más bajas se concentran principalmente en las personas de 60 años en adelante. Risaralda presenta una tasa del 0,33 %, seguido de Meta, con un 1,18 %; Chocó, con un 1,21 %; Cauca, con un 1,27 %, y Magdalena, con un 1,40 %.
Una posible explicación es que algunas personas de este grupo de edad se retiran del mercado laboral y pasan a la población económicamente inactiva. Como la tasa de desempleo se calcula únicamente sobre la población económicamente activa, quienes dejan de buscar empleo no se contabilizan como desempleados. Por esta razón, una tasa baja no significa necesariamente que todas las personas mayores tengan mejores condiciones laborales.

Los resultados muestran que el grupo de 18 a 24 años presenta las mayores tasas de desempleo en 25 de los 33 departamentos analizados. Por otro lado, las personas de 24 a 30 años registran las tasas más altas en siete departamentos, entre ellos Amazonas, Arauca, Boyacá, Chocó, Nariño, Putumayo y Vaupés.
Asimismo, las diferencias entre grupos de edad pueden ser considerables. En Bolívar, por ejemplo, la tasa de desempleo de los jóvenes de 18 a 24 años es del 30,86 %, mientras que para las personas de 60 años en adelante es del 2,98 %, una diferencia de 27,88 puntos porcentuales.

El desempleo en Colombia presenta diferencias importantes según la edad y el departamento. Los jóvenes son el grupo que enfrenta mayores dificultades para conseguir empleo en la mayoría de los territorios analizados, mientras que los adultos mayores registran tasas considerablemente inferiores. Estos resultados resaltan la importancia de fortalecer las oportunidades de inserción laboral juvenil y de analizar las condiciones particulares de cada región. También es necesario interpretar las tasas con precaución, debido a que no muestran por sí solas la calidad del empleo ni la situación de quienes están fuera del mercado laboral.

4. ¿Cuáles son los 3 departamentos con menor tasa de desempleo de mujeres entre los 18 y los 25 años ?  

In [0]:
# Mujeres desempleadas entre 18 y 25 años por departamento
desempleadas_jovenes = desempleados[
    (desempleados['DSI'] == 1) &
    (desempleados['P3271'] == 2) &
    (desempleados['P6040'].between(18, 25))
].groupby('DPTO')['FEX_C18'].sum()

# Mujeres económicamente activas entre 18 y 25 años por departamento
pea_mujeres_jovenes = ft[
    (ft['FT'] == 1) &
    (ft['P3271'] == 2) &
    (ft['P6040'].between(18, 25))
].groupby('DPTO')['FEX_C18'].sum()

# Crear tabla con los resultados
tabla_mujeres_jovenes = pd.DataFrame({
    'PEA': pea_mujeres_jovenes,
    'DESEMPLEADAS': desempleadas_jovenes
})

# Completar con cero donde no se registraron desempleadas
tabla_mujeres_jovenes['DESEMPLEADAS'] = (
    tabla_mujeres_jovenes['DESEMPLEADAS'].fillna(0)
)

# Conservar departamentos con población económicamente activa
tabla_mujeres_jovenes = tabla_mujeres_jovenes[
    tabla_mujeres_jovenes['PEA'] > 0
].copy()

# Calcular la tasa de desempleo
tabla_mujeres_jovenes['TD'] = (
    tabla_mujeres_jovenes['DESEMPLEADAS'] /
    tabla_mujeres_jovenes['PEA']
) * 100

# Agregar los nombres de los departamentos
tabla_mujeres_jovenes = tabla_mujeres_jovenes.reset_index()
tabla_mujeres_jovenes['Departamento'] = (
    tabla_mujeres_jovenes['DPTO'].map(dpto_nombre)
)

# Excluir Bogotá, porque es un distrito y se solicitan departamentos
tabla_mujeres_jovenes = tabla_mujeres_jovenes[
    tabla_mujeres_jovenes['DPTO'] != 11
]

# Seleccionar los tres departamentos con menor tasa de desempleo
resultado_punto4 = tabla_mujeres_jovenes.nsmallest(
    3, 'TD'
)[['Departamento', 'PEA', 'DESEMPLEADAS', 'TD']].reset_index(drop=True)

display(resultado_punto4)

# Respuesta escrita con los resultados obtenidos
print('Los tres departamentos con menor tasa de desempleo '
      'de mujeres entre 18 y 25 años son:')

for _, fila in resultado_punto4.iterrows():
    print(f"- {fila['Departamento']}: {fila['TD']:.2f}%.")

print(
    '\nEstas tasas representan el porcentaje de mujeres desocupadas '
    'dentro de las mujeres económicamente activas de 18 a 25 años '
    'en cada departamento. Una tasa menor no implica necesariamente '
    'mayor participación laboral ni mejores condiciones de empleo.'
)

Los tres departamentos identificados presentan las menores tasas de desempleo entre las mujeres de 18 a 25 años. Esto indica que, dentro de este grupo, una menor proporción de quienes participan en el mercado laboral se encuentra desocupada, en comparación con los demás departamentos analizados.

5. Calcule el promedio de ingresos laborales (variable INGLABO) de las personas residentes en los departamentos de Risaralda, Caldas , Quindío, Antioquia y Valle del Cauca. 

In [0]:
# Leer la tabla de ocupados
ocupados = spark.read.format("delta").table(
    "workspace.electiva.ocupados"
).toPandas()

# Departamentos: Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca
departamentos = [66, 17, 63, 5, 76]

# Seleccionar personas ocupadas de los departamentos solicitados
ingresos_dpto = ocupados[
    (ocupados['OCI'] == 1) &
    (ocupados['DPTO'].isin(departamentos))
].copy()

# Convertir ingresos y factor de expansión a valores numéricos
ingresos_dpto['INGLABO'] = pd.to_numeric(
    ingresos_dpto['INGLABO'], errors='coerce'
)

ingresos_dpto['FEX_C18'] = pd.to_numeric(
    ingresos_dpto['FEX_C18'], errors='coerce'
)

# Conservar ingresos registrados y factores positivos
ingresos_dpto = ingresos_dpto[
    ingresos_dpto['INGLABO'].notna() &
    (ingresos_dpto['FEX_C18'] > 0)
].copy()

# Calcular el ingreso ponderado
ingresos_dpto['INGRESO_PONDERADO'] = (
    ingresos_dpto['INGLABO'] * ingresos_dpto['FEX_C18']
)

# Sumar los ingresos ponderados y los factores por departamento
tabla_ingresos = ingresos_dpto.groupby('DPTO')[
    ['INGRESO_PONDERADO', 'FEX_C18']
].sum().reset_index()

# Calcular el promedio de ingresos laborales
tabla_ingresos['Promedio INGLABO'] = (
    tabla_ingresos['INGRESO_PONDERADO'] /
    tabla_ingresos['FEX_C18']
)

# Agregar nombres de departamentos
tabla_ingresos['Departamento'] = (
    tabla_ingresos['DPTO'].map(dpto_nombre)
)

# Seleccionar columnas y ordenar de mayor a menor ingreso
resultado_punto5 = tabla_ingresos[
    ['Departamento', 'Promedio INGLABO']
].sort_values(
    by='Promedio INGLABO', ascending=False
).reset_index(drop=True)

display(resultado_punto5)



El promedio de ingresos laborales fue de $2.465.234,67 en Antioquia, $2.224.140,05 en Caldas, $2.209.455,00 en Risaralda, $2.170.380,88 en Quindío y $2.122.049,96 en Valle del Cauca.
Entre los cinco departamentos analizados, Antioquia registró el mayor ingreso laboral promedio y Valle del Cauca el menor, con una diferencia de $343.184,71. Estos resultados evidencian diferencias en los ingresos laborales promedio entre los departamentos, aunque no permiten determinar por sí solos sus causas ni cómo se distribuyen los ingresos entre las personas.